# Official MUVERA: short WANDS diagnostic
One configuration, 10 queries, full corpus. Uses Google's pinned C++ FDE generator and your saved LateOn tokens. **CPU runtime is sufficient.** No model download, document re-encoding, or serving benchmark.

The first run compiles native code and generates document FDEs. Keep this runtime open: subsequent query/budget changes reuse the caches. Only compact reports are backed up to Drive. This tests official FDE generation, not the complete paper serving stack.

In [ ]:
from pathlib import Path
import collections, os, subprocess, sys, time
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/ras-wands-official-fde')
LOGS = Path('/content/drive/MyDrive/ras_wands_official_fde_logs')
LOGS.mkdir(parents=True, exist_ok=True)

def run_logged(command, name):
    tail = collections.deque(maxlen=50)
    last = time.monotonic()
    print(f'Starting {name}; log: {LOGS / (name + ".log")}', flush=True)
    with (LOGS / (name + '.log')).open('a') as log:
        with subprocess.Popen(command, cwd=REPO if REPO.exists() else '/content',
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    log.write(line); log.flush(); tail.append(line)
                    if name == 'diagnostic' or time.monotonic()-last > 20 or 'FAILED' in line or 'passed' in line or 'OFFICIAL_FDE_READY' in line:
                        print(line, end='', flush=True)
                        last = time.monotonic()
                result = process.wait()
            except BaseException:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
                raise
    if result:
        raise RuntimeError(f'{name} failed: {LOGS / (name + ".log")}\n' + ''.join(tail))
    print(f'Finished {name}', flush=True)

BRANCH = 'codex/colbert-muvera-baselines'
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)], 'clone')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], 'update')
os.environ.update(PYTHONPATH=str(REPO / 'src') + ':' + str(REPO),
                  OPENBLAS_NUM_THREADS='2', OMP_NUM_THREADS='2', USE_TF='0', USE_FLAX='0')
# Install only the diagnostic/build dependencies. No torch, PyLate or model downloads.
run_logged([sys.executable, '-m', 'pip', 'install', 'cmake>=3.24,<4', 'ninja',
            'numpy', 'pandas', 'threadpoolctl', 'ir-measures==0.4.3', 'pytest'], 'install')
BUILD = Path('/content/ras-google-fde-build')
run_logged([sys.executable, '-m', 'experiments.build_google_fde',
            '--build-dir', str(BUILD), '--jobs', '2'], 'native_build')
os.environ['GOOGLE_FDE_LIBRARY'] = str(BUILD / 'libgoogle_fde.so')
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_google_fde.py'], 'native_tests')


## Stage existing inputs once
The token file is approximately 8.2 GB. If the earlier local caches are still present, they are reused. No old FDE indexes or model files are copied. After a runtime reset this initial copy and native build must run again.

In [ ]:
import json, shutil
REFERENCE_DRIVE = Path('/content/drive/MyDrive/ras_wands_denseon_lateon_seed7_v1')
CACHE_DRIVE = Path('/content/drive/MyDrive/ras_wands_muvera_cost_seed7_v1')
REFERENCE = Path('/content/wands_systems_reference')
CACHE = Path('/content/ras_wands_muvera_cost_seed7_v1')

def stage(source, target, names):
    copied = 0
    for name in names:
        src, dst = source/name, target/name
        if not src.is_file(): raise FileNotFoundError(f'Missing saved experiment input: {src}')
        if dst.exists() and dst.stat().st_size == src.stat().st_size and abs(dst.stat().st_mtime-src.stat().st_mtime) < 2:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.stat().st_size > 100_000_000 or copied % 50 == 0:
            print(f'Staging inputs: {copied} copied; current {name} ({src.stat().st_size/1e6:.1f} MB)', flush=True)
        temporary = dst.with_suffix(dst.suffix + '.copying')
        shutil.copy2(src, temporary)
        temporary.replace(dst)
        copied += 1
    print(f'Staging complete: {copied} files copied; remaining files reused')

names = ['manifest.json', 'dataset.json', 'candidate_ids.json']
for arm in ['dense', 'colbert']:
    scores = sorted((REFERENCE_DRIVE/arm).glob('scores_*.npy'))
    if not scores: raise FileNotFoundError(f'No saved full scores in {REFERENCE_DRIVE/arm}')
    names += [str(p.relative_to(REFERENCE_DRIVE)) for p in scores]
stage(REFERENCE_DRIVE, REFERENCE, names)
stage(CACHE_DRIVE, CACHE, ['manifest.json', 'parity.json', 'serving.json',
      'colbert/values.npy', 'colbert/offsets.npy', 'colbert/queries.npz'])


## Short loop — rerun this cell
Default: official R=20, 5 partition bits, 8-dimensional **upstream sparse inner projection**, producing 5,120-dimensional FDEs. Full-corpus candidate search is exact inner product; saved exact LateOn scores rerank candidates.

Change budgets without recomputing FDEs or query scores. Change query count to reuse document FDEs. Change configuration to build a new document FDE cache. The default 10 queries are a prefix of the earlier seeded 60-query diagnostic sample; no held-out claims.

In [ ]:
import hashlib
CONFIG = 'r20_b5_p8' # @param ['r20_b5_p8', 'r20_b5_p16', 'r8_b4_4096', 'raw_r8_b4']
QUERIES = 10 # @param {type:'integer'}
CANDIDATES = [100, 1000, 5000]
FDE_SEED = 7
QUERY_SEED = 7
code_revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
settings = dict(config=CONFIG, queries=QUERIES, candidates=CANDIDATES, fde_seed=FDE_SEED, query_seed=QUERY_SEED, code_revision=code_revision)
tag = CONFIG + '_q' + str(QUERIES) + '_' + hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:8]
RUN = Path('/content/ras_wands_official_quick') / tag
BACKUP = Path('/content/drive/MyDrive/ras_wands_official_quick') / tag
FDE_CACHE = Path('/content/ras_wands_official_fde_cache')
command = [sys.executable, '-u', '-m', 'experiments.wands_google_fde',
    '--reference-dir', str(REFERENCE), '--cache-dir', str(CACHE),
    '--output-dir', str(RUN), '--fde-cache-dir', str(FDE_CACHE),
    '--library', str(BUILD/'libgoogle_fde.so'), '--config', CONFIG,
    '--queries', str(QUERIES), '--query-seed', str(QUERY_SEED), '--fde-seed', str(FDE_SEED),
    '--candidates', *map(str, CANDIDATES), '--threads', '2']
try:
    run_logged(command, 'diagnostic')
finally:
    BACKUP.mkdir(parents=True, exist_ok=True)
    reports = [p for p in RUN.glob('*') if p.suffix in ['.csv', '.json']]
    for p in reports: shutil.copy2(p, BACKUP/p.name)
    print(f'Saved {len(reports)} compact reports to {BACKUP}')


In [ ]:
import pandas as pd
from IPython.display import display
assert json.loads((RUN/'complete.json').read_text())['status'] == 'complete'
print('Candidate fidelity: top-k recall measures agreement with exact LateOn, not human relevance.')
display(pd.read_csv(RUN/'fidelity.csv'))
print('Quality: exact LateOn, dense and candidate rerankers on the same queries and full-corpus judgments.')
display(pd.read_csv(RUN/'quality.csv'))
print('This is an offline diagnostic. No new serving latency or memory measurement.')
